In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor
%env TENSORLY_BACKEND=numpy

import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'


print(f'TensorLy backend: {tl.get_backend()}')
print(f'TensorLy tenalg backend: {tl.tenalg.get_backend()}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUPY_ACCELERATORS=cutensor
env: TENSORLY_BACKEND=numpy
TensorLy backend: numpy
TensorLy tenalg backend: core


In [2]:
from moabb.paradigms import FilterBankMotorImagery, MotorImagery
from moabb.datasets import *
from hoda.tensorize import fh_power, fh_log_envelope
from hoda.classification import ZLogRatio, ZScore
dataset = Schirrmeister2017()
dataset.event_id




{'right_hand': 1, 'left_hand': 2, 'rest': 3, 'feet': 4}

In [3]:
events = list(dataset.event_id.keys())[:3]
events

['right_hand', 'left_hand', 'rest']

In [4]:
sfreq=250
paradigm = MotorImagery(events=events, n_classes=3, resample=sfreq)
paradigm.used_events(dataset)

{'right_hand': 1, 'left_hand': 2, 'rest': 3}

In [ ]:
from sklearn.preprocessing import FunctionTransformer
import sys
sys.path.append('../')
from classification_mi import stf_transform 

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[1],
     return_epochs=False,
     postprocess_pipeline = FunctionTransformer(stf_transform),
     cache_config=cache_config,

)



/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/download.py:55: RuntimeWarning:

Setting non-standard config type: "MNE_DATASETS_SCHIRRMEISTER2017_PATH"

/usr/local/share/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning:

Unverified HTTPS request is being made to host 'web.gin.g-node.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings

/usr/local/share/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning:

Unverified HTTPS request is being made to host 'gin.g-node.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings

0.00B [00:00, ?B/s]     
SHA256 hash of downloaded file: 943390216871aee03dac6cda77e0f0ba34bc9adfc9d8bc7790127981b13b7bc4
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file ha

Extracting EDF parameters from /home/arne/mne_data/MNE-schirrmeister2017-data/robintibor/high-gamma-dataset/raw/master/data/train/1.edf...
EDF file detected
Channel 'EEG Fp1' recognized as type EEG (renamed to 'Fp1').
Channel 'EEG Fp2' recognized as type EEG (renamed to 'Fp2').
Channel 'EEG Fpz' recognized as type EEG (renamed to 'Fpz').
Channel 'EEG F7' recognized as type EEG (renamed to 'F7').
Channel 'EEG F3' recognized as type EEG (renamed to 'F3').
Channel 'EEG Fz' recognized as type EEG (renamed to 'Fz').
Channel 'EEG F4' recognized as type EEG (renamed to 'F4').
Channel 'EEG F8' recognized as type EEG (renamed to 'F8').
Channel 'EEG FC5' recognized as type EEG (renamed to 'FC5').
Channel 'EEG FC1' recognized as type EEG (renamed to 'FC1').
Channel 'EEG FC2' recognized as type EEG (renamed to 'FC2').
Channel 'EEG FC6' recognized as type EEG (renamed to 'FC6').
Channel 'EEG M1' recognized as type EEG (renamed to 'M1').
Channel 'EEG T7' recognized as type EEG (renamed to 'T7').
Cha

/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning:

warnEpochs <Epochs | 240 events (all good), 0 – 4 s (baseline off), ~469.1 MiB, data loaded,
 'right_hand': 80
 'left_hand': 80
 'rest': 80>

/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning:

This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.

/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning:

warnEpochs <Epochs | 120 events (all good), 0 – 4 s (baseline off), ~234.6 MiB, data loaded,
 'right_hand': 40
 'left_hand': 40
 'rest': 40>

/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning:

This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transf

In [ ]:
X.shape

In [ ]:
import numpy as np
np.unique(y)

In [ ]:
X = tl.tensor(X)
X = ZScore().fit_transform(X)

In [ ]:
import plotly.express as px
import numpy as np

px.imshow(np.cov(tl.to_numpy(tl.unfold(X,1))))

In [ ]:
px.imshow(np.cov(tl.to_numpy(tl.unfold(X,2))))

In [ ]:
px.imshow(np.cov(tl.to_numpy(tl.unfold(X,3))), color_continuous_scale='RdBu_r', zmin=-1.5, zmax=1.5)

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import scipy.stats as stats
import matplotlib
%matplotlib inline

measurements = np.random.normal(loc = 20, scale = 5, size=100)   
stats.probplot(tl.to_numpy(X).flatten(), dist="norm", plot=plt)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#x_mean = tl.to_numpy(X_tfr_base[y=='feet']).mean(axis=0)
#for f in range(x_mean.shape[0]):
#    sns.heatmap(x_mean[f], cmap='RdBu_r', center=0, square='True')
#    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import pandas as pd

x = tl.to_numpy(X.flatten())
hist, bin_edges = np.histogram(x, bins=100000)
bins = (bin_edges[:-1] + bin_edges[1:]) / 2  # Compute the mean of each subsequent pair
df = pd.DataFrame({'bins':bins, 'count':hist})
px.histogram(df, x="bins", y="count")

In [ ]:
from hoda.hoda import HODA

hoda = HODA(
        rank=None,
        max_iter=256,
        tol=1e-6,
        shrinkage='lw',
        toeplitz=None,
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        forward=True,
        theta=0.5,
        refit_shrinkage=True,
)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from hoda.util import solve_gevdh
from hoda.cov import mode_scatter, ledoit_wolf_shrinkage

plt.style.use('default')
%load_ext line_profiler

hoda.fit_backward(X,y)
df = pd.DataFrame(hoda.train_info_['backward'])
display(df)

In [ ]:
from hoda.classification import SelectFCutoff
Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt, 0))
print(xt.shape)
select = SelectFCutoff()
select.fit(xt,y)
print(select.support_)

In [ ]:
# 0.270325 s, 42.8%

In [ ]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[0]))

In [ ]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[1]))

In [ ]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[2]))

In [ ]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

In [ ]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr', log_y=False)
    fig.show()

In [ ]:
px.line(df, x='iteration', y='update', log_y=True, color='mode')


In [ ]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [ ]:
px.line(df, x='flip', y='objective', log_y=False, color='mode')

In [ ]:
%load_ext line_profiler
%lprun -f hoda.fit_forward hoda.fit_forward(X,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

In [ ]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

In [ ]:
if hoda.extra_train_info:
    x.line(df, x='flip', y='mse', log_y=True)

In [ ]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [ ]:
for i in range(hoda.weights_[1].shape[1]):
    plt.plot(tl.to_numpy(hoda.aps_[1][:,i]))
    plt.show()

In [ ]:
for i in range(hoda.aps_[1].shape[1]):
    plt.plot(tl.to_numpy(hoda.aps_[1][:,i]))
    plt.show()

In [ ]:
for i in range(hoda.weights_[2].shape[1]):
    plt.plot(tl.to_numpy(hoda.weights_[2][:,i]))
    plt.show()

In [ ]:
for i in range(hoda.aps_[2].shape[1]):
    plt.plot(tl.to_numpy(hoda.aps_[2][:,i]))
    plt.show()

In [ ]:
from sklearn.feature_selection import f_classif
import numpy as np
from sklearn.decomposition import PCA
Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))
F, p = f_classif(xt, y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': F,
    'selected': F>1
})
fig = px.bar(df, x='feature', y='F', color='selected', log_y=True)
fig.show()

In [ ]:
from sklearn.feature_selection import f_classif
import numpy as np
from sklearn.decomposition import PCA
Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))
F, p = f_classif(xt, y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': F,
    'selected': F>1
})
fig = px.bar(df, x='feature', y='F', color='selected', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [ ]:
Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))


xt_pca = PCA(whiten=True, n_components=None).fit_transform(xt)
F_pca, p = f_classif(xt_pca, y)

df = pd.DataFrame({
    'feature': np.arange(len(F_pca)),
    'F': F_pca,
    'selected': F_pca>1
})
fig = px.bar(df, y='F', color='selected', log_y=True)
fig.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


fig = px.scatter(x=xt_pca[:,0], y=xt_pca[:,1], color=y)
fig.update_layout(width=1000, height=800)

### 